# Phase 21 — probabilistic multiscale regime models

This notebook compares contemporaneous clustering, an independent Gaussian mixture, and causal-filtered HMMs against the frozen direct taxonomy. The direct taxonomy is a comparator, not a supervised target or hidden truth.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    REGIME_ORDER,
    build_multiscale_state,
    candidate_summary,
    filtering_prefix_invariant,
    fit_candidate,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
)

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
PRIMARY_CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
N_STATES = 4
RANDOM_STATE = 17
MAX_ITER = 40
N_RESTARTS = 2
RUN_BAR_TIME_HMM_SENSITIVITY = True
LAST_N = 10_000

## Frozen comparison contract

All models use four states for this first comparison because the direct baseline has four structural classes. State semantics are mapped using the **training partition only**. Agreement with the direct taxonomy measures relationship, not accuracy against truth.

- K-means: contemporaneous hard clustering; no temporal state.
- Gaussian mixture: contemporaneous soft membership; no Markov history.
- Normal HMM: Gaussian emissions plus filtered Markov history.
- Student-t HMM: heavier-tailed emissions plus filtered Markov history.

The primary clock is union-event time. A Normal-HMM bar-time sensitivity is run separately because changing clocks changes the population and transition meaning.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, PRIMARY_CLOCK, FEATURE_TIER)

split_summary = pd.DataFrame.from_records([
    {"split": name, "rows": len(frame), "start": frame.index[0], "end": frame.index[-1]}
    for name, frame in prepared.frames.items()
]).set_index("split")
display(split_summary)
print(f"Features: {len(prepared.feature_columns)}")
print(f"Train end: {prepared.split.train_end}; validation end: {prepared.split.validation_end}")

,rows,start,end
split,,,
train,2045,2023-05-25 09:35:00+02:00,2025-06-23 04:30:00+02:00
validation,557,2025-06-23 07:10:00+02:00,2026-03-02 23:25:00+02:00
test,522,2026-03-03 11:45:00+02:00,2026-11-09 10:15:00+02:00


Features: 36
Train end: 2025-06-23 05:55:00+02:00; validation end: 2026-03-03 07:20:00+02:00


## Fit all predeclared candidates

Scaling is fitted on training rows only. HMM probabilities are forward-filtered. Validation and test state semantics use the immutable mapping learned from the training contingency table.

In [3]:
candidate_families = (
    ModelFamily.KMEANS,
    ModelFamily.GAUSSIAN_MIXTURE,
    ModelFamily.NORMAL_HMM,
    ModelFamily.STUDENTT_HMM,
)
fits = {}
summary_records = []
for family in candidate_families:
    fit = fit_candidate(
        prepared, family, N_STATES, RANDOM_STATE, max_iter=MAX_ITER, n_restarts=N_RESTARTS
    )
    fits[family] = fit
    record = candidate_summary(fit, prepared)
    record["filtering_prefix_invariant"] = filtering_prefix_invariant(fit, prepared)
    summary_records.append(record)

candidate_table = pd.DataFrame.from_records(summary_records).set_index("family")
display(candidate_table.style.format({
    "validation_log_likelihood_per_row": "{:.4f}",
    "minimum_state_occupancy": "{:.2%}",
    "switch_rate": "{:.2%}",
    "median_dwell": "{:.1f}",
    "mean_max_probability": "{:.2%}",
    "baseline_adjusted_rand": "{:.3f}",
    "baseline_nmi": "{:.3f}",
    "baseline_mapped_agreement": "{:.2%}",
}))

,n_states,clock,tier,error,validation_log_likelihood_per_row,minimum_state_occupancy,switch_rate,median_dwell,mean_max_probability,baseline_adjusted_rand,baseline_nmi,baseline_mapped_agreement,filtering_prefix_invariant
family,,,,,,,,,,,,,
kmeans,4,union_event_time,dynamic,None,nan,22.26%,67.81%,1.0,31.86%,0.708,0.781,75.94%,None
gaussian_mixture,4,union_event_time,dynamic,None,-18.4397,12.93%,64.21%,1.0,99.90%,0.909,0.904,86.71%,None
normal_hmm,4,union_event_time,dynamic,None,-23.7706,10.95%,32.91%,2.0,99.74%,0.350,0.520,54.22%,True
studentt_hmm,4,union_event_time,dynamic,None,-18.9370,13.46%,57.01%,1.0,100.00%,0.297,0.438,50.09%,True


## State occupancy and training-only semantic maps

A model state mapped to `fragmented` means that its training observations overlapped that direct class most strongly under the one-to-one assignment. The model state remains its own probabilistic object.

In [4]:
map_records = []
occupancy_records = []
for family, fit in fits.items():
    if fit.error is not None:
        continue
    for state, regime_code in fit.semantic_map.items():
        map_records.append({
            "family": family.value,
            "raw_state": state,
            "training_semantic_map": REGIME_ORDER[regime_code],
        })
    for split_name, states in fit.states.items():
        counts = np.bincount(states, minlength=N_STATES) / len(states)
        for state, share in enumerate(counts):
            occupancy_records.append({
                "family": family.value, "split": split_name, "state": state, "share": share
            })
display(pd.DataFrame.from_records(map_records).set_index(["family", "raw_state"]))
occupancy = pd.DataFrame.from_records(occupancy_records).pivot_table(
    index=["family", "state"], columns="split", values="share"
)
display(occupancy.style.format("{:.2%}"))

training_semantic_map
family           raw_state                      
kmeans           0                    fragmented
                 1               broad_coherence
                 2                  deep_counter
                 3               shallow_counter
gaussian_mixture 0                    fragmented
                 1                  deep_counter
                 2               shallow_counter
                 3               broad_coherence
normal_hmm       0               shallow_counter
                 1                  deep_counter
                 2                    fragmented
                 3               broad_coherence
studentt_hmm     0               broad_coherence
                 1                    fragmented
                 2                  deep_counter
                 3               shallow_counter

## Observation-clock sensitivity

This is not a search for the clock with the most attractive labels. It asks how much persistence is introduced when the same evolving hierarchy is sampled on every bar instead of only on union-DCC bars.

In [ ]:
clock_sensitivity = []
if RUN_BAR_TIME_HMM_SENSITIVITY:
    bar_prepared = prepare_data(global_state, ObservationClock.BAR, FEATURE_TIER, split=prepared.split)
    bar_fit = fit_candidate(
        bar_prepared, ModelFamily.NORMAL_HMM, N_STATES, RANDOM_STATE, MAX_ITER, N_RESTARTS
    )
    event_record = candidate_summary(fits[ModelFamily.NORMAL_HMM], prepared)
    bar_record = candidate_summary(bar_fit, bar_prepared)
    event_record["filtering_prefix_invariant"] = filtering_prefix_invariant(
        fits[ModelFamily.NORMAL_HMM], prepared
    )
    bar_record["filtering_prefix_invariant"] = filtering_prefix_invariant(bar_fit, bar_prepared)
    clock_sensitivity = pd.DataFrame.from_records([event_record, bar_record]).set_index("clock")
    display(clock_sensitivity)
else:
    print("Bar-time HMM sensitivity disabled by configuration.")

## Recent filtered HMM states on price

The displayed state is post-observation and causal at each union-event timestamp. It is not smoothed backward. Visually similar observations may receive different states because filtered prior probability is part of the HMM decision.

In [ ]:
successful_hmms = [
    fit for family, fit in fits.items()
    if family in {ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM} and fit.error is None
]
if not successful_hmms:
    raise RuntimeError("No HMM candidate completed. Inspect candidate_table.error.")
display_fit = max(
    successful_hmms,
    key=lambda fit: fit.log_likelihoods["validation"] / len(prepared.frames["validation"]),
)
display_parts = []
for split_name in ("train", "validation", "test"):
    part = prepared.frames[split_name].copy()
    part["model_state"] = display_fit.states[split_name]
    part["model_semantic"] = [REGIME_ORDER[code] for code in display_fit.mapped_regimes[split_name]]
    for state in range(display_fit.n_states):
        part[f"filtered_p_state_{state}"] = display_fit.probabilities[split_name][:, state]
    display_parts.append(part)
display_frame = pd.concat(display_parts).sort_index()
probability_columns = [f"filtered_p_state_{state}" for state in range(display_fit.n_states)]
plot_price_states(
    display_frame,
    "model_semantic",
    f"{SYMBOL}: {display_fit.family.value} causal filtered regimes",
    probability_columns=probability_columns,
    last_n=LAST_N,
).show()

# Phase 21 completion gate

1. K-means and the mixture classify the current observation; only the HMM carries filtered state history.
2. Filtered probabilities, not smoothed probabilities, are the deployable HMM output.
3. Training-only semantic mapping prevents validation/test relabelling.
4. Agreement with the direct taxonomy is descriptive because the taxonomy is not truth.
5. A high-persistence or high-confidence model is not automatically useful.
6. Bar-time and event-time HMMs model different transition processes.
7. Model selection is deferred to the predeclared causal/OOS phase.